In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 8 — O TEOREMA DE BAYES: A SABEDORIA DA PROBABILIDADE

> "A probabilidade não nos diz o que vai acontecer. Ela nos diz o que é sensato acreditar que vai acontecer, dada a evidência que temos."
> — E. T. Jaynes, *Teoria da Probabilidade: A Lógica da Ciência*

Depois de semanas preparando os dados, eu me sentia como um artesão que afiara todas as ferramentas. Agora precisava do meu primeiro soldado. E eu não queria começar com uma caixa-preta impenetrável: minha jornada era sobre transparência. Precisava de um algoritmo cuja lógica eu pudesse seguir.

Meu pensamento voltou ao século XVIII, ao **Reverendo Thomas Bayes**. Seu teorema oferece uma forma de atualizar crenças à luz de novas evidências — não é isso que um diagnóstico faz? O médico parte de uma crença inicial e a atualiza com os exames. O classificador **Naive Bayes**, derivado desse teorema, era o ponto de partida perfeito: simples, rápido e surpreendentemente competente.

## Naive Bayes: A Lógica da Probabilidade em Ação

O Naive Bayes calcula a probabilidade de um evento (o tumor ser maligno) *dado* um conjunto de evidências (as *features*). O teorema:

**P(A | B) = [ P(B | A) · P(A) ] / P(B)**

No nosso caso, queremos **P(Diagnóstico | Features)**. A parte "ingênua" (*naive*) é uma suposição forte: o algoritmo assume que **todas as *features* são independentes entre si, dada a classe**. Sabemos, da análise de correlação do Capítulo 5, que isso é falso — `mean radius` e `mean area` são altamente correlacionadas. E, no entanto, o Naive Bayes costuma funcionar muito bem mesmo com a suposição violada. É um excelente ponto de partida.

> **📘 PARA SABER — Sabores de Naive Bayes**
>
> **Gaussian** — assume *features* contínuas com distribuição normal. É o que usaremos.
>
> **Multinomial** — para contagens discretas (ex.: frequência de palavras em texto).
>
> **Bernoulli** — para *features* binárias.

## Nosso Primeiro Classificador Real

Treinamos um **Gaussian Naive Bayes** e o avaliamos por validação cruzada no treino. Reportamos duas métricas: a **acurácia** e — a que de fato importa clinicamente — o **recall da classe maligna**, importado pronto do módulo do projeto.

Um detalhe técnico importante: ao contrário de SVM ou KNN, o Gaussian Naive Bayes **não é sensível à escala** das *features* (ele modela a distribuição de cada uma isoladamente). Por isso, aqui, dispensamos o `StandardScaler`.

#### Código 8.1: Treinando um Gaussian Naive Bayes


In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_validate
from oncoclassify_utils import X_train, y_train, recall_maligno

# recall_maligno = recall da classe 0 (Maligno), o alvo clinico do projeto.
resultado = cross_validate(GaussianNB(), X_train, y_train, cv=5,
                           scoring={"acuracia": "accuracy", "recall_Maligno": recall_maligno})

print(f"Acuracia media:        {resultado['test_acuracia'].mean():.4f}")
print(f"Recall Maligno medio:  {resultado['test_recall_Maligno'].mean():.4f}")

Acuracia media:        0.9437
Recall Maligno medio:  0.9037


Um salto enorme em relação ao baseline (63,3%): **94,37%** de acurácia com um dos algoritmos mais simples do arsenal. Mais relevante para nós, o **recall da classe maligna é 90,37%** — o modelo encontra nove em cada dez cânceres. Ainda longe do ideal clínico, mas um começo sólido e, sobretudo, **honesto**: é o recall da classe que importa, não uma média que dilui o risco.

> **💡 DICA — Nem todo modelo precisa de escala**
>
> Padronizar *features* é essencial para SVM, KNN e Regressão Logística, mas **inócuo** para o Gaussian Naive Bayes e para modelos baseados em árvores. Saber quando escalar (e quando não) evita passos desnecessários no *pipeline* e mal-entendidos sobre o que cada algoritmo realmente faz.

> **📌 CHECKLIST DO CAPÍTULO 8**
>
> ✅ Entende a intuição do Teorema de Bayes aplicada à classificação.
>
> ✅ Compreende a suposição "ingênua" da independência condicional.
>
> ✅ Sabe que o Gaussian Naive Bayes **não** é sensível à escala.
>
> ✅ Executou o Código 8.1 e obteve **acurácia 94,4%** e **recall maligno 90,4%**.
>
> **⚠️ CRÍTICO:** Estas são estimativas obtidas **só no treino**, para comparar algoritmos. O teste de fogo no conjunto de teste (`X_test`) só acontece no fim do projeto.

Os números no terminal eram mais do que estatísticas: eram um sopro de esperança. Com um dos algoritmos mais simples, eu já superava de longe o desempenho implícito do OncoClassify 1.0. A simplicidade, ao que parecia, era poderosa.

Mas esta era apenas a primeira arma. O Naive Bayes pensava em probabilidades; agora eu queria uma abordagem que pensasse em **geometria** — que traçasse a linha de separação mais clara e ousada possível entre os dois mundos. Era hora de invocar as Máquinas de Vetores de Suporte.
